# September 28 final Gini run — sanity check

This notebook compares the **final Gini-regularized model**

\[
\lambda_G = 10^{-4}
\]

against the finalized September 17 **unregularized** model.

It is deliberately a **diagnostic / before-vs-after analyzer**, not yet the
final paper analyzer. Its purpose is to answer:

1. Did Gini regularization improve recovery of the weighted influence matrix \(A\)?
2. Did prediction quality improve or deteriorate?
3. How much did downstream control change?

The comparison is paired by the same environment and learning seed whenever
possible. The environment is then used as the primary unit of analysis by
averaging the three learning seeds within environment.

For the original single-shot run, \(A\)-recovery is loaded from the previously
validated deterministic-replay analysis:

`results/final_A_recovery_analysis_2026_09_19/A_recovery_seed_level.csv`

For the September-28 runs, \(A\)-recovery metrics are stored directly by the
computation notebooks, so no replay or training is performed here.

**No model fitting or environment rollout occurs in this notebook.**


In [21]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import t as student_t

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 320)
pd.set_option("display.max_rows", 300)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError(
        "Could not find repository root containing opinion_dynamics/."
    )


REPO_ROOT = find_repo_root()
RESULTS_ROOT = (
    REPO_ROOT / "opinion_dynamics" / "experiments" / "results"
)

OLD_SS_STUDY = (
    "broad_all_dynamics_multitopology_multiseed_unbounded_lambda1"
)
OLD_AB_STUDY = (
    "abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1"
)
OLD_SS_PIPELINE = "2026-09-17-unbounded-lambda1-v1"
OLD_AB_PIPELINE = "2026-09-17-unbounded-lambda1-v1"
OLD_SS_CONFIG_HASH = (
    "8521f6ff6ad1c49c2b9cdf0ff6cb143b751dc282bf65f7be6e62336b41643afa"
)
OLD_AB_CONFIG_HASH = "2dfd8a85437a16ee"

NEW_SS_STUDY = (
    "broad_all_dynamics_multitopology_multiseed_unbounded_lambda1_gini1e4"
)
NEW_AB_STUDY = (
    "abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1_gini1e4"
)
NEW_PIPELINE = "2026-09-28-unbounded-lambda1-gini1e4-v1"
EXPECTED_REGULARIZER = "gini"
EXPECTED_REGULARIZER_LAMBDA = 1e-4

LEARNING_SEEDS = [0, 1, 2]
EXPECTED_ENVS = 195
EXPECTED_LEARNED = 585

OLD_SS_RESULTS = (
    RESULTS_ROOT
    / "experiment_2026_09_17_broad_all_dynamics_multitopology_multiseed_unbounded_lambda1"
)
OLD_AB_RESULTS = (
    RESULTS_ROOT
    / "experiment_2026_09_17_abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1"
)

# The generated Sep-28 computation notebooks accidentally duplicated the
# suffix in RESULTS_DIR. Accept the actual folder first, while also allowing
# the intended clean folder name if it was manually corrected before running.
NEW_SS_CANDIDATES = [
    RESULTS_ROOT
    / "experiment_2026_09_28_broad_all_dynamics_multitopology_multiseed_unbounded_lambda1_gini1e4_gini1e4",
    RESULTS_ROOT
    / "experiment_2026_09_28_broad_all_dynamics_multitopology_multiseed_unbounded_lambda1_gini1e4",
]
NEW_AB_CANDIDATES = [
    RESULTS_ROOT
    / "experiment_2026_09_28_abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1_gini1e4_gini1e4",
    RESULTS_ROOT
    / "experiment_2026_09_28_abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1_gini1e4",
]

OLD_A_RECOVERY = (
    RESULTS_ROOT
    / "final_A_recovery_analysis_2026_09_19"
    / "A_recovery_seed_level.csv"
)

ANALYSIS_DIR = RESULTS_ROOT / "final_gini_sanity_analysis_2026_09_28"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Analysis output:", ANALYSIS_DIR)


Repository: d:\Work\repos\RL\unknown_graph_networks
Analysis output: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\final_gini_sanity_analysis_2026_09_28


## Resolve and validate the four source computations

In [22]:
def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def resolve_completed_result_dir(
    candidates: list[Path],
    *,
    expected_study: str,
) -> Path:
    completed = []
    for path in candidates:
        completion_path = path / "COMPUTATION_COMPLETE.json"
        if not completion_path.exists():
            continue
        try:
            completion = read_json(completion_path)
        except Exception:
            continue
        if (
            completion.get("status") == "success"
            and completion.get("study_name") == expected_study
        ):
            completed.append(path)

    if len(completed) == 0:
        raise FileNotFoundError(
            "No completed result directory found for "
            f"{expected_study}. Checked:\n"
            + "\n".join(f"  - {p}" for p in candidates)
        )
    if len(completed) > 1:
        raise RuntimeError(
            "More than one completed result directory matches the same "
            f"study. Refusing an ambiguous analysis:\n"
            + "\n".join(f"  - {p}" for p in completed)
        )
    return completed[0]


NEW_SS_RESULTS = resolve_completed_result_dir(
    NEW_SS_CANDIDATES,
    expected_study=NEW_SS_STUDY,
)
NEW_AB_RESULTS = resolve_completed_result_dir(
    NEW_AB_CANDIDATES,
    expected_study=NEW_AB_STUDY,
)

for path in [OLD_SS_RESULTS, OLD_AB_RESULTS, NEW_SS_RESULTS, NEW_AB_RESULTS]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Old single-shot:", OLD_SS_RESULTS)
print("New single-shot:", NEW_SS_RESULTS)
print("Old abundant:", OLD_AB_RESULTS)
print("New abundant:", NEW_AB_RESULTS)


FileNotFoundError: No completed result directory found for abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1_gini1e4. Checked:
  - d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\experiment_2026_09_28_abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1_gini1e4_gini1e4
  - d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\experiment_2026_09_28_abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1_gini1e4

In [ ]:
# Old final single-shot provenance.
old_ss_manifest = read_json(OLD_SS_RESULTS / "RUN_MANIFEST.json")
old_ss_completion = read_json(
    OLD_SS_RESULTS / "COMPUTATION_COMPLETE.json"
)

assert old_ss_manifest["study_name"] == OLD_SS_STUDY
assert old_ss_manifest["pipeline_version"] == OLD_SS_PIPELINE
assert old_ss_manifest["config_hash"] == OLD_SS_CONFIG_HASH
assert int(old_ss_manifest["n_environments"]) == EXPECTED_ENVS
assert int(old_ss_manifest["n_learned_runs_expected"]) == EXPECTED_LEARNED
assert list(old_ss_manifest["learning_seeds"]) == LEARNING_SEEDS

assert old_ss_completion["status"] == "success"
assert old_ss_completion["study_name"] == OLD_SS_STUDY
assert old_ss_completion["config_hash"] == OLD_SS_CONFIG_HASH

# Old final abundant provenance.
old_ab_completion = read_json(
    OLD_AB_RESULTS / "COMPUTATION_COMPLETE.json"
)
assert old_ab_completion["status"] == "success"
assert old_ab_completion["study_name"] == OLD_AB_STUDY
assert old_ab_completion["config_hash"] == OLD_AB_CONFIG_HASH
assert int(old_ab_completion["n_environments"]) == EXPECTED_ENVS
assert int(old_ab_completion["n_learned_runs"]) == EXPECTED_LEARNED

# New final single-shot provenance.
new_ss_manifest = read_json(NEW_SS_RESULTS / "RUN_MANIFEST.json")
new_ss_completion = read_json(
    NEW_SS_RESULTS / "COMPUTATION_COMPLETE.json"
)

assert new_ss_manifest["study_name"] == NEW_SS_STUDY
assert new_ss_manifest["pipeline_version"] == NEW_PIPELINE
assert int(new_ss_manifest["n_environments"]) == EXPECTED_ENVS
assert int(new_ss_manifest["n_learned_runs_expected"]) == EXPECTED_LEARNED
assert list(new_ss_manifest["learning_seeds"]) == LEARNING_SEEDS

new_ss_cfg = new_ss_manifest["scientific_config"]
assert float(new_ss_cfg["lambda_mix"]) == 1.0
assert new_ss_cfg["regularizer_kind"] == EXPECTED_REGULARIZER
assert np.isclose(
    float(new_ss_cfg["regularizer_lambda"]),
    EXPECTED_REGULARIZER_LAMBDA,
)

assert new_ss_completion["status"] == "success"
assert new_ss_completion["study_name"] == NEW_SS_STUDY
assert new_ss_completion["pipeline_version"] == NEW_PIPELINE
assert new_ss_completion["regularizer_kind"] == EXPECTED_REGULARIZER
assert np.isclose(
    float(new_ss_completion["regularizer_lambda"]),
    EXPECTED_REGULARIZER_LAMBDA,
)
assert int(new_ss_completion["n_environments"]) == EXPECTED_ENVS
assert int(new_ss_completion["n_learned_runs"]) == EXPECTED_LEARNED

# New final abundant provenance.
new_ab_completion = read_json(
    NEW_AB_RESULTS / "COMPUTATION_COMPLETE.json"
)
assert new_ab_completion["status"] == "success"
assert new_ab_completion["study_name"] == NEW_AB_STUDY
assert new_ab_completion["pipeline_version"] == NEW_PIPELINE
assert new_ab_completion["regularizer_kind"] == EXPECTED_REGULARIZER
assert np.isclose(
    float(new_ab_completion["regularizer_lambda"]),
    EXPECTED_REGULARIZER_LAMBDA,
)
assert new_ab_completion["source_dataset_study"] == OLD_AB_STUDY
assert new_ab_completion["source_dataset_config_hash"] == OLD_AB_CONFIG_HASH
assert int(new_ab_completion["n_environments"]) == EXPECTED_ENVS
assert int(new_ab_completion["n_learned_runs"]) == EXPECTED_LEARNED
assert int(new_ab_completion["n_combined_rows"]) == EXPECTED_ENVS * 6

print("STRICT PROVENANCE CHECKS PASSED")
print("New SS config hash:", new_ss_completion["config_hash"])
print("New SS implementation hash:", new_ss_completion["implementation_hash"])
print("New AB config hash:", new_ab_completion["config_hash"])
print("New AB implementation hash:", new_ab_completion["implementation_hash"])


## Load single-shot cache summaries

In [ ]:
def resolve_manifest_cache_dir(
    manifest: dict[str, Any],
    row: dict[str, Any],
) -> Path:
    original = Path(row["cache_dir"])
    if original.exists():
        return original

    cache_root = Path(manifest["cache_root"])
    fallback = cache_root / original.name
    if fallback.exists():
        return fallback

    raise FileNotFoundError(
        f"Environment cache not found for {row['environment_id']}:\n"
        f"  manifest path: {original}\n"
        f"  fallback: {fallback}"
    )


def load_single_shot_run(
    manifest: dict[str, Any],
    *,
    learned_policy: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    baseline_rows = []
    learned_rows = []

    for env in manifest["environments"]:
        env_dir = resolve_manifest_cache_dir(manifest, env)

        baseline_path = env_dir / "baseline_summary.json"
        if not baseline_path.exists():
            raise FileNotFoundError(baseline_path)
        baseline_rows.extend(read_json(baseline_path))

        learned_root = env_dir / "learned"
        if not learned_root.exists():
            raise FileNotFoundError(learned_root)

        for seed in LEARNING_SEEDS:
            matches = sorted(
                learned_root.glob(f"seed_{seed:02d}_*/summary.json")
            )
            if len(matches) != 1:
                raise RuntimeError(
                    f"{env['environment_id']}, seed={seed}: expected one "
                    f"learned summary, found {len(matches)}"
                )
            row = read_json(matches[0])
            if row.get("policy") != learned_policy:
                raise RuntimeError(
                    f"{matches[0]}: policy={row.get('policy')!r}, "
                    f"expected {learned_policy!r}"
                )
            learned_rows.append(row)

    baseline = pd.DataFrame(baseline_rows)
    learned = pd.DataFrame(learned_rows)

    assert len(baseline) == EXPECTED_ENVS * 3
    assert len(learned) == EXPECTED_LEARNED
    assert learned["environment_id"].nunique() == EXPECTED_ENVS
    assert learned.groupby("environment_id").size().eq(3).all()

    return baseline, learned


old_ss_baseline, old_ss_learned = load_single_shot_run(
    old_ss_manifest,
    learned_policy="learned_nonlinear",
)
new_ss_baseline, new_ss_learned = load_single_shot_run(
    new_ss_manifest,
    learned_policy="learned_nonlinear_gini",
)

print("Old SS learned:", old_ss_learned.shape)
print("New SS learned:", new_ss_learned.shape)
display(
    new_ss_learned[
        [
            "environment_id",
            "learning_seed",
            "final_mean",
            "final_fit_model_over_identity",
            "row_TV",
            "true_edge_mass",
            "edge_AP",
        ]
    ].head()
)


## Load abundant summaries and verify exact dataset reuse

In [ ]:
old_ab = pd.read_csv(OLD_AB_RESULTS / "combined_summary.csv")
new_ab = pd.read_csv(NEW_AB_RESULTS / "combined_summary.csv")

old_ab_baseline = old_ab[
    old_ab["policy"].isin(["no_control", "uniform", "true_graph"])
].copy()
old_ab_learned = old_ab[
    old_ab["policy"] == "learned_abundant_nonlinear"
].copy()

new_ab_baseline = new_ab[
    new_ab["policy"].isin(["no_control", "uniform", "true_graph"])
].copy()
new_ab_learned = new_ab[
    new_ab["policy"] == "learned_abundant_nonlinear_gini"
].copy()

assert len(old_ab_baseline) == EXPECTED_ENVS * 3
assert len(new_ab_baseline) == EXPECTED_ENVS * 3
assert len(old_ab_learned) == EXPECTED_LEARNED
assert len(new_ab_learned) == EXPECTED_LEARNED

assert new_ab_learned["regularizer_kind"].eq(
    EXPECTED_REGULARIZER
).all()
assert np.allclose(
    new_ab_learned["regularizer_lambda"].astype(float),
    EXPECTED_REGULARIZER_LAMBDA,
)

new_ab_cache_root = (
    RESULTS_ROOT
    / "_trial_cache"
    / NEW_AB_STUDY
    / str(new_ab_completion["config_hash"])
)

dataset_audit = []
for env_id in sorted(new_ab_learned["environment_id"].unique()):
    meta_path = new_ab_cache_root / env_id / "dataset_meta.json"
    if not meta_path.exists():
        raise FileNotFoundError(meta_path)
    meta = read_json(meta_path)
    dataset_audit.append(
        {
            "environment_id": env_id,
            "dataset_reused_exactly": bool(
                meta.get("dataset_reused_exactly", False)
            ),
            "source_dataset_study": meta.get("source_dataset_study"),
            "source_dataset_config_hash": meta.get(
                "source_dataset_config_hash"
            ),
            "n_pairs": int(meta.get("n_pairs", -1)),
        }
    )

dataset_audit = pd.DataFrame(dataset_audit)

assert len(dataset_audit) == EXPECTED_ENVS
assert dataset_audit["dataset_reused_exactly"].all()
assert dataset_audit["source_dataset_study"].eq(OLD_AB_STUDY).all()
assert dataset_audit["source_dataset_config_hash"].eq(
    OLD_AB_CONFIG_HASH
).all()
assert dataset_audit["n_pairs"].eq(2500).all()

print(
    "ABUNDANT DATASET REUSE AUDIT PASSED: "
    "195/195 exact paired datasets."
)
print("Old AB learned:", old_ab_learned.shape)
print("New AB learned:", new_ab_learned.shape)


## Baseline integrity check

The regularizer only changes the learned identifier. The no-control, uniform,
and true-graph baselines should therefore remain unchanged.

This is an end-to-end check that the benchmark environment definitions were not
accidentally modified between September 17 and September 28.


In [ ]:
def baseline_delta_table(
    old_df: pd.DataFrame,
    new_df: pd.DataFrame,
    *,
    metric: str,
    regime: str,
) -> pd.DataFrame:
    keys = ["environment_id", "policy"]
    old = old_df[keys + [metric]].rename(
        columns={metric: "old_value"}
    )
    new = new_df[keys + [metric]].rename(
        columns={metric: "new_value"}
    )
    pair = old.merge(new, on=keys, validate="one_to_one")
    pair["abs_delta"] = np.abs(
        pair["new_value"] - pair["old_value"]
    )
    return (
        pair.groupby("policy", as_index=False)
        .agg(
            n=("environment_id", "size"),
            max_abs_delta=("abs_delta", "max"),
            mean_abs_delta=("abs_delta", "mean"),
        )
        .assign(regime=regime)
    )


baseline_audit = pd.concat(
    [
        baseline_delta_table(
            old_ss_baseline,
            new_ss_baseline,
            metric="final_mean",
            regime="single_shot",
        ),
        baseline_delta_table(
            old_ab_baseline,
            new_ab_baseline,
            metric="mean_end",
            regime="abundant",
        ),
    ],
    ignore_index=True,
)

display(baseline_audit)

if baseline_audit["max_abs_delta"].max() > 1e-10:
    print(
        "NOTE: baselines are not bit-identical. Inspect the table above "
        "before interpreting learned-model changes."
    )
else:
    print(
        "BASELINE INTEGRITY PASSED: final values are identical "
        "to numerical precision."
    )


## Load the validated September-17 A-recovery reference

In [ ]:
if not OLD_A_RECOVERY.exists():
    raise FileNotFoundError(
        f"Missing old A-recovery table: {OLD_A_RECOVERY}\n"
        "Run the validated "
        "analyze_final_A_recovery_STRICT_FAST_ANALYSIS.ipynb once."
    )

old_rec = pd.read_csv(OLD_A_RECOVERY)

required_old_rec = {
    "regime",
    "environment_id",
    "scenario_class",
    "family",
    "dynamics",
    "learning_seed",
    "A_MAE",
    "A_Fro",
    "row_TV",
    "true_edge_mass",
    "support_topdegree_recall",
    "edge_AP",
    "centrality_L1",
}
missing = sorted(required_old_rec - set(old_rec.columns))
if missing:
    raise RuntimeError(
        f"Old A-recovery table is missing columns: {missing}"
    )

old_ss_rec = old_rec[
    old_rec["regime"] == "single_shot"
].copy()
old_ab_rec = old_rec[
    old_rec["regime"] == "abundant"
].copy()

assert len(old_ss_rec) == EXPECTED_LEARNED
assert len(old_ab_rec) == EXPECTED_LEARNED

print("Old A-recovery rows:")
display(
    old_rec.groupby(
        ["regime", "scenario_class", "dynamics"]
    ).size().rename("runs").reset_index()
)


## Canonicalize old/new learned results

In [ ]:
RECOVERY_METRICS = [
    "A_MAE",
    "A_Fro",
    "row_TV",
    "true_edge_mass",
    "support_topdegree_recall",
    "edge_AP",
    "centrality_L1",
]

KEYS = [
    "environment_id",
    "scenario_class",
    "family",
    "dynamics",
    "topology_seed",
    "opinion_seed",
    "environment_seed",
    "learning_seed",
]


def canonical_performance(
    df: pd.DataFrame,
    *,
    regime: str,
    control_col: str,
    prediction_col: str,
) -> pd.DataFrame:
    keep = KEYS + [control_col, prediction_col]
    result = df[keep].copy()
    result = result.rename(
        columns={
            control_col: "control_final",
            prediction_col: "prediction_ratio",
        }
    )
    result.insert(0, "regime", regime)
    return result


old_perf = pd.concat(
    [
        canonical_performance(
            old_ss_learned,
            regime="single_shot",
            control_col="final_mean",
            prediction_col="final_fit_model_over_identity",
        ),
        canonical_performance(
            old_ab_learned,
            regime="abundant",
            control_col="mean_end",
            prediction_col="model_over_identity",
        ),
    ],
    ignore_index=True,
)

new_perf = pd.concat(
    [
        canonical_performance(
            new_ss_learned,
            regime="single_shot",
            control_col="final_mean",
            prediction_col="final_fit_model_over_identity",
        ),
        canonical_performance(
            new_ab_learned,
            regime="abundant",
            control_col="mean_end",
            prediction_col="model_over_identity",
        ),
    ],
    ignore_index=True,
)

old_recovery = old_rec[
    ["regime"] + KEYS + RECOVERY_METRICS
].copy()

new_recovery = pd.concat(
    [
        new_ss_learned[KEYS + RECOVERY_METRICS].assign(
            regime="single_shot"
        ),
        new_ab_learned[KEYS + RECOVERY_METRICS].assign(
            regime="abundant"
        ),
    ],
    ignore_index=True,
)

assert len(old_perf) == 2 * EXPECTED_LEARNED
assert len(new_perf) == 2 * EXPECTED_LEARNED
assert len(old_recovery) == 2 * EXPECTED_LEARNED
assert len(new_recovery) == 2 * EXPECTED_LEARNED

print("Canonical old/new rows:", len(old_perf), len(new_perf))


## Exact paired seed-level comparison

Positive control delta means the regularized model controls better.

For error metrics (`prediction_ratio`, `A_MAE`, `A_Fro`, `row_TV`,
`centrality_L1`), **negative** deltas mean improvement.

For `true_edge_mass`, `support_topdegree_recall`, and `edge_AP`, **positive**
deltas mean improvement.


In [ ]:
pair_keys = ["regime"] + KEYS

perf_pair = old_perf.merge(
    new_perf,
    on=pair_keys,
    suffixes=("_old", "_new"),
    validate="one_to_one",
)

rec_pair = old_recovery.merge(
    new_recovery,
    on=pair_keys,
    suffixes=("_old", "_new"),
    validate="one_to_one",
)

# perf_pair and rec_pair each already contain the metadata columns; merge only
# the recovery value columns into the performance pairing.
rec_value_cols = (
    pair_keys
    + [f"{m}_old" for m in RECOVERY_METRICS]
    + [f"{m}_new" for m in RECOVERY_METRICS]
)

seed_pair = perf_pair.merge(
    rec_pair[rec_value_cols],
    on=pair_keys,
    validate="one_to_one",
)

assert len(seed_pair) == 2 * EXPECTED_LEARNED

for metric in ["control_final", "prediction_ratio"] + RECOVERY_METRICS:
    seed_pair[f"delta_{metric}"] = (
        seed_pair[f"{metric}_new"]
        - seed_pair[f"{metric}_old"]
    )

for regime, g in seed_pair.groupby("regime"):
    assert g["environment_id"].nunique() == EXPECTED_ENVS
    assert g.groupby("environment_id").size().eq(3).all()

seed_pair.to_csv(
    ANALYSIS_DIR / "gini_vs_unregularized_seed_level.csv",
    index=False,
)

print("PAIRED SEED-LEVEL AUDIT PASSED:", len(seed_pair), "rows")


## Environment-level comparison — primary analysis unit

In [ ]:
meta_cols = [
    "regime",
    "environment_id",
    "scenario_class",
    "family",
    "dynamics",
    "topology_seed",
    "opinion_seed",
    "environment_seed",
]

metric_names = ["control_final", "prediction_ratio"] + RECOVERY_METRICS

numeric_cols = (
    [f"{m}_old" for m in metric_names]
    + [f"{m}_new" for m in metric_names]
    + [f"delta_{m}" for m in metric_names]
)

env_pair = (
    seed_pair.groupby(
        meta_cols,
        as_index=False,
        dropna=False,
    )[numeric_cols]
    .mean()
)

assert len(env_pair) == 2 * EXPECTED_ENVS
assert env_pair.groupby("regime").size().eq(EXPECTED_ENVS).all()

env_pair.to_csv(
    ANALYSIS_DIR / "gini_vs_unregularized_environment_level.csv",
    index=False,
)

print("Environment-level paired rows:", len(env_pair))


## Overall headline comparison

In [ ]:
def overall_summary(g: pd.DataFrame) -> dict[str, float]:
    return {
        "n_environments": int(len(g)),
        "control_old": g["control_final_old"].mean(),
        "control_new": g["control_final_new"].mean(),
        "delta_control": g["delta_control_final"].mean(),

        "prediction_old": g["prediction_ratio_old"].mean(),
        "prediction_new": g["prediction_ratio_new"].mean(),
        "delta_prediction": g["delta_prediction_ratio"].mean(),

        "row_TV_old": g["row_TV_old"].mean(),
        "row_TV_new": g["row_TV_new"].mean(),
        "delta_row_TV": g["delta_row_TV"].mean(),

        "true_edge_mass_old": g["true_edge_mass_old"].mean(),
        "true_edge_mass_new": g["true_edge_mass_new"].mean(),
        "delta_true_edge_mass": g["delta_true_edge_mass"].mean(),

        "support_recall_old": g["support_topdegree_recall_old"].mean(),
        "support_recall_new": g["support_topdegree_recall_new"].mean(),
        "delta_support_recall": g["delta_support_topdegree_recall"].mean(),

        "edge_AP_old": g["edge_AP_old"].mean(),
        "edge_AP_new": g["edge_AP_new"].mean(),
        "delta_edge_AP": g["delta_edge_AP"].mean(),

        "centrality_L1_old": g["centrality_L1_old"].mean(),
        "centrality_L1_new": g["centrality_L1_new"].mean(),
        "delta_centrality_L1": g["delta_centrality_L1"].mean(),

        "fraction_env_row_TV_improved": (
            g["delta_row_TV"] < 0
        ).mean(),
        "fraction_env_edge_mass_improved": (
            g["delta_true_edge_mass"] > 0
        ).mean(),
        "fraction_env_control_improved": (
            g["delta_control_final"] > 0
        ).mean(),
    }


headline = pd.DataFrame(
    [
        {"regime": regime, **overall_summary(g)}
        for regime, g in env_pair.groupby("regime")
    ]
)

display(headline.round(6))
headline.to_csv(
    ANALYSIS_DIR / "headline_gini_vs_unregularized.csv",
    index=False,
)


## Uncertainty by dynamics

For generic \(5\times5\) topology/opinion experiments, confidence intervals use
a crossed bootstrap: topology and initial-opinion seeds are resampled
independently.

For each structured dynamics block (15 environments), paired Student-\(t\)
intervals are used.

This matches the statistical treatment used in the validated final paper
analysis rather than treating the three learning seeds as independent samples.


In [ ]:
def student_t_ci(
    values: np.ndarray,
    confidence: float = 0.95,
) -> tuple[float, float, float]:
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    mean = float(x.mean())
    if len(x) <= 1:
        return mean, np.nan, np.nan
    crit = float(
        student_t.ppf(
            0.5 + confidence / 2,
            df=len(x) - 1,
        )
    )
    half = crit * float(x.std(ddof=1)) / math.sqrt(len(x))
    return mean, mean - half, mean + half


def crossed_bootstrap_ci(
    group: pd.DataFrame,
    col: str,
    *,
    n_boot: int = 20_000,
    seed: int = 20260928,
) -> tuple[float, float, float]:
    pivot = group.pivot(
        index="topology_seed",
        columns="opinion_seed",
        values=col,
    )
    if pivot.shape != (5, 5) or pivot.isna().any().any():
        raise ValueError(
            f"Incomplete crossed design for {col}: {pivot.shape}"
        )

    matrix = pivot.to_numpy(float)
    rng = np.random.default_rng(seed)

    row_idx = rng.integers(0, 5, size=(n_boot, 5))
    col_idx = rng.integers(0, 5, size=(n_boot, 5))

    sampled = matrix[
        row_idx[:, :, None],
        col_idx[:, None, :],
    ]
    boot = sampled.mean(axis=(1, 2))
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return float(matrix.mean()), float(lo), float(hi)


def paired_ci(
    group: pd.DataFrame,
    col: str,
) -> tuple[float, float, float, str]:
    if (
        group["scenario_class"].iloc[0] == "unstructured_random"
        and len(group) == 25
        and group["topology_seed"].nunique() == 5
        and group["opinion_seed"].nunique() == 5
    ):
        mean, lo, hi = crossed_bootstrap_ci(group, col)
        return mean, lo, hi, "crossed bootstrap"

    mean, lo, hi = student_t_ci(group[col].to_numpy())
    return mean, lo, hi, "paired Student-t"


analysis_metrics = [
    ("control_final", "higher"),
    ("prediction_ratio", "lower"),
    ("row_TV", "lower"),
    ("true_edge_mass", "higher"),
    ("support_topdegree_recall", "higher"),
    ("edge_AP", "higher"),
    ("centrality_L1", "lower"),
]

rows = []

for (
    regime,
    scenario_class,
    dynamics,
), g in env_pair.groupby(
    ["regime", "scenario_class", "dynamics"],
    sort=False,
):
    for metric, direction in analysis_metrics:
        delta_col = f"delta_{metric}"
        mean, lo, hi, method = paired_ci(g, delta_col)
        rows.append(
            {
                "regime": regime,
                "scenario_class": scenario_class,
                "dynamics": dynamics,
                "metric": metric,
                "better_direction": direction,
                "old_mean": g[f"{metric}_old"].mean(),
                "new_mean": g[f"{metric}_new"].mean(),
                "delta_new_minus_old": mean,
                "ci_low": lo,
                "ci_high": hi,
                "ci_method": method,
                "n_environments": len(g),
            }
        )

by_dynamics = pd.DataFrame(rows)

by_dynamics.to_csv(
    ANALYSIS_DIR / "gini_vs_unregularized_by_dynamics.csv",
    index=False,
)

for metric in [
    "control_final",
    "prediction_ratio",
    "row_TV",
    "true_edge_mass",
    "edge_AP",
]:
    print("\n", "=" * 100)
    print(metric)
    print("=" * 100)
    display(
        by_dynamics[
            by_dynamics["metric"] == metric
        ][
            [
                "regime",
                "scenario_class",
                "dynamics",
                "old_mean",
                "new_mean",
                "delta_new_minus_old",
                "ci_low",
                "ci_high",
                "ci_method",
            ]
        ].round(6)
    )


## Structured HK stress case

In [ ]:
hk = by_dynamics[
    (by_dynamics["scenario_class"] == "structured_mechanism")
    & (by_dynamics["dynamics"] == "hegselmannkrause")
].copy()

display(
    hk[
        [
            "regime",
            "metric",
            "better_direction",
            "old_mean",
            "new_mean",
            "delta_new_minus_old",
            "ci_low",
            "ci_high",
        ]
    ].round(6)
)


## Per-dynamics compact table

In [ ]:
def compact_dynamics_table(
    regime: str,
) -> pd.DataFrame:
    g = by_dynamics[
        by_dynamics["regime"] == regime
    ].copy()

    wanted = [
        "control_final",
        "prediction_ratio",
        "row_TV",
        "true_edge_mass",
        "edge_AP",
        "centrality_L1",
    ]

    records = []
    for (
        scenario_class,
        dynamics,
    ), block in g.groupby(
        ["scenario_class", "dynamics"],
        sort=False,
    ):
        row = {
            "scenario_class": scenario_class,
            "dynamics": dynamics,
        }
        for metric in wanted:
            m = block[
                block["metric"] == metric
            ].iloc[0]
            row[f"{metric}_old"] = m["old_mean"]
            row[f"{metric}_new"] = m["new_mean"]
            row[f"{metric}_delta"] = m[
                "delta_new_minus_old"
            ]
        records.append(row)

    return pd.DataFrame(records)


for regime in ["single_shot", "abundant"]:
    print("\n", "=" * 100)
    print(regime)
    print("=" * 100)
    table = compact_dynamics_table(regime)
    display(table.round(6))
    table.to_csv(
        ANALYSIS_DIR
        / f"compact_{regime}_gini_vs_unregularized.csv",
        index=False,
    )


## How often does recovery improve?

This section avoids reducing the result to only a grand mean. It reports how
frequently the regularized model improves individual environment instances
after averaging the three neural-network seeds.


In [ ]:
frequency_rows = []

for regime, g in env_pair.groupby("regime"):
    frequency_rows.append(
        {
            "regime": regime,
            "n_environments": len(g),
            "row_TV_better": int(
                (g["delta_row_TV"] < 0).sum()
            ),
            "row_TV_better_fraction": (
                g["delta_row_TV"] < 0
            ).mean(),
            "true_edge_mass_better": int(
                (g["delta_true_edge_mass"] > 0).sum()
            ),
            "true_edge_mass_better_fraction": (
                g["delta_true_edge_mass"] > 0
            ).mean(),
            "edge_AP_better": int(
                (g["delta_edge_AP"] > 0).sum()
            ),
            "edge_AP_better_fraction": (
                g["delta_edge_AP"] > 0
            ).mean(),
            "prediction_better": int(
                (g["delta_prediction_ratio"] < 0).sum()
            ),
            "prediction_better_fraction": (
                g["delta_prediction_ratio"] < 0
            ).mean(),
            "control_better": int(
                (g["delta_control_final"] > 0).sum()
            ),
            "control_better_fraction": (
                g["delta_control_final"] > 0
            ).mean(),
        }
    )

frequency = pd.DataFrame(frequency_rows)
display(frequency.round(4))


## Control-change distribution

In [ ]:
control_quantiles = (
    env_pair.groupby("regime")["delta_control_final"]
    .quantile([0.05, 0.25, 0.50, 0.75, 0.95])
    .unstack()
    .rename(
        columns={
            0.05: "q05",
            0.25: "q25",
            0.50: "median",
            0.75: "q75",
            0.95: "q95",
        }
    )
    .reset_index()
)

display(control_quantiles.round(6))


## Recovery improvement versus control change

In [ ]:
for regime in ["single_shot", "abundant"]:
    g = env_pair[
        env_pair["regime"] == regime
    ]

    fig, ax = plt.subplots(figsize=(7.0, 5.0))
    ax.axhline(0.0, linewidth=1.0, linestyle="--")
    ax.axvline(0.0, linewidth=1.0, linestyle="--")

    ax.scatter(
        g["delta_row_TV"],
        g["delta_control_final"],
        alpha=0.65,
    )

    ax.set_xlabel(
        "Δ row-TV (Gini − unregularized; negative = better A recovery)"
    )
    ax.set_ylabel(
        "Δ final control (Gini − unregularized; positive = better)"
    )
    ax.set_title(
        f"{regime}: A-recovery change vs control change"
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)


## Mean deltas by dynamics

In [ ]:
for regime in ["single_shot", "abundant"]:
    g = (
        env_pair[env_pair["regime"] == regime]
        .groupby(
            ["scenario_class", "dynamics"],
            as_index=False,
        )
        .agg(
            delta_row_TV=("delta_row_TV", "mean"),
            delta_control=("delta_control_final", "mean"),
        )
    )
    g["label"] = (
        g["scenario_class"].str.replace(
            "_", " ", regex=False
        )
        + " | "
        + g["dynamics"]
    )

    fig, ax = plt.subplots(figsize=(8.0, 5.0))
    ax.barh(
        g["label"],
        g["delta_row_TV"],
    )
    ax.axvline(0.0, linewidth=1.0)
    ax.set_xlabel(
        "Mean Δ row-TV (negative = improved A recovery)"
    )
    ax.set_title(
        f"{regime}: Gini effect on weighted-A recovery"
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8.0, 5.0))
    ax.barh(
        g["label"],
        g["delta_control"],
    )
    ax.axvline(0.0, linewidth=1.0)
    ax.set_xlabel(
        "Mean Δ final control (positive = improved control)"
    )
    ax.set_title(
        f"{regime}: Gini effect on downstream control"
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)


## Diagnostic summary

This section deliberately prints observations rather than applying an arbitrary
accept/reject threshold. The important quantities for the decision are:

- whether mean and per-dynamics `row_TV` move downward;
- whether `true_edge_mass` moves upward;
- whether support ranking (`edge_AP`, support recall) is preserved or traded
  away;
- the magnitude and distribution of `delta_control`;
- whether prediction ratio improves;
- whether the structured-HK pathology improves.


In [ ]:
for _, row in headline.iterrows():
    regime = row["regime"]
    print("=" * 100)
    print(regime.upper())
    print("=" * 100)
    print(
        f"Control:          {row['control_old']:.6f} -> "
        f"{row['control_new']:.6f}  "
        f"(Δ {row['delta_control']:+.6f})"
    )
    print(
        f"Prediction ratio: {row['prediction_old']:.6f} -> "
        f"{row['prediction_new']:.6f}  "
        f"(Δ {row['delta_prediction']:+.6f}; lower is better)"
    )
    print(
        f"row-TV:           {row['row_TV_old']:.6f} -> "
        f"{row['row_TV_new']:.6f}  "
        f"(Δ {row['delta_row_TV']:+.6f}; lower is better)"
    )
    print(
        f"True-edge mass:   {row['true_edge_mass_old']:.6f} -> "
        f"{row['true_edge_mass_new']:.6f}  "
        f"(Δ {row['delta_true_edge_mass']:+.6f})"
    )
    print(
        f"Edge AP:          {row['edge_AP_old']:.6f} -> "
        f"{row['edge_AP_new']:.6f}  "
        f"(Δ {row['delta_edge_AP']:+.6f})"
    )
    print(
        f"Centrality L1:    {row['centrality_L1_old']:.6f} -> "
        f"{row['centrality_L1_new']:.6f}  "
        f"(Δ {row['delta_centrality_L1']:+.6f}; lower is better)"
    )
    print(
        "Environment fraction with improved row-TV: "
        f"{row['fraction_env_row_TV_improved']:.1%}"
    )
    print(
        "Environment fraction with improved true-edge mass: "
        f"{row['fraction_env_edge_mass_improved']:.1%}"
    )
    print(
        "Environment fraction with improved control: "
        f"{row['fraction_env_control_improved']:.1%}"
    )
    print()

print("Outputs written to:", ANALYSIS_DIR)
print("GINI FINAL-RUN SANITY ANALYSIS COMPLETE")
